# Get to Know a Dataset: GeoTessera Geospatial Embeddings

This notebook is a 101-level guided tour of the **GeoTessera geospatial embeddings** dataset (Tessera foundation model embeddings derived from Sentinel-1/2). The dataset landing page is on the [Registry of Open Data on AWS](https://registry.opendata.aws/geotessera-embeddings).

**What you will learn**
- How the S3 bucket is organized (prefix structure)
- What file formats are included and how to interpret them
- How to download a single tile from public S3 and dequantize it
- A few simple visualizations and sanity-check statistics for exploratory analysis


## Dataset overview

GeoTessera provides access to **precomputed geospatial embeddings** from the [Tessera foundation model](https://github.com/ucam-eo/tessera). The model ingests a year of Sentinel-1 and Sentinel-2 observations and produces a dense **128-channel** representation map at **10m** resolution.

**Key characteristics**
- **Spatial tiling**: global 0.1° × 0.1° tiles, named by **tile center coordinates** (e.g., `grid_0.15_52.05`)
- **Temporal coverage**: multiple years (e.g., 2017–2024, depending on availability)
- **Data storage**: embeddings are stored as **quantized int8** arrays with **per-pixel scale factors** for dequantization

Dequantization rule:

$$
\texttt{embedding\_float32} = \texttt{embedding\_int8.astype(float32)} \times \texttt{scales\_float32}
$$

For a higher-level view of how the dataset is used in practice (registry lookup, tile selection, etc.), see the main project README in the GeoTessera repository.


In [ ]:
# Requirements (requirements.txt format)
# boto3
# botocore
# numpy
# matplotlib
#
# Optional (only if you want to read GeoTIFF landmask tiles locally):
# rasterio
#
# This tutorial uses anonymous (unsigned) S3 access to a public bucket.

In [ ]:
import re
from io import BytesIO

import boto3
import numpy as np
import matplotlib.pyplot as plt

from botocore import UNSIGNED
from botocore.config import Config


## Connect to the public S3 bucket

We will access the dataset directly from S3 using **anonymous / unsigned** requests. Update the bucket name below if the Registry entry points to a different bucket.


In [ ]:
bucket = "geotessera-open-data"

s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
)

# List top-level prefixes / objects
resp = s3.list_objects_v2(Bucket=bucket, Delimiter="/")
for p in resp.get("CommonPrefixes", []):
    print("PREFIX:", p["Prefix"])
for obj in resp.get("Contents", []):
    print("OBJECT:", obj["Key"])


## Dataset organization (S3 prefix structure)

At the top level, the S3 bucket is organized around a small number of human-readable prefixes:

- `global_0.1_degree_representation/` — quantized embedding tiles and associated scale files, organized by year (e.g., `2017/` ... `2024/`).
- `global_0.1_degree_tiff_all/` — landmask GeoTIFF tiles used for georeferencing and native projection information.

Within `global_0.1_degree_representation/<YEAR>/`, each 0.1° × 0.1° tile (named by its **center coordinates**) typically includes two NumPy files:

- `grid_<lon>_<lat>.npy` — **int8** quantized embeddings of shape `(height, width, 128)`.
- `grid_<lon>_<lat>_scales.npy` — **float32** scale factors for dequantization (broadcastable to the embedding array).

Landmask tiles in `global_0.1_degree_tiff_all/` are GeoTIFF files named `grid_<lon>_<lat>.tiff` and provide georeferencing information (projection and affine transform) for the corresponding tile.

Tile naming uses the **tile center**. A tile with center `(lon, lat)` covers:
- Longitude: `[lon - 0.05°, lon + 0.05°]`
- Latitude: `[lat - 0.05°, lat + 0.05°]`


## Explore the prefix structure

Let's list which years are available under `global_0.1_degree_representation/`.


In [ ]:
repr_prefix = "global_0.1_degree_representation/"
resp = s3.list_objects_v2(Bucket=bucket, Prefix=repr_prefix, Delimiter="/", MaxKeys=1000)

years = []
for p in resp.get("CommonPrefixes", []):
    m = re.match(r"global_0\.1_degree_representation/(\d{4})/", p["Prefix"])
    if m:
        years.append(int(m.group(1)))

print("Available years:", sorted(years))


Pick one year (prefer 2024 if available) and list a few tile files to understand naming conventions.


In [ ]:
year = 2024 if 2024 in years else (max(years) if years else 2024)

year_prefix = f"{repr_prefix}{year}/"
resp = s3.list_objects_v2(Bucket=bucket, Prefix=year_prefix, MaxKeys=200)
keys = [obj["Key"] for obj in resp.get("Contents", [])]

# Show a few example keys
for k in sorted([k for k in keys if k.endswith(".npy")])[:20]:
    print(k)

print("Shown", min(20, len([k for k in keys if k.endswith('.npy')])), "example .npy objects from", year_prefix)


## Data formats

- **Embeddings**: `*.npy` files containing **int8** quantized embeddings with shape `(height, width, 128)`.
- **Scales**: `*_scales.npy` files containing **float32** scale factors. These are broadcastable to the embedding shape so you can compute dequantized embeddings efficiently.
- **Landmasks (georeferencing)**: `*.tiff` GeoTIFF tiles providing the native projection (typically UTM zone) and affine transform for the tile.

**Advice**
- Use NumPy to load `.npy` objects quickly.
- For most ML workflows, you can work purely in array space. Only bring in GeoTIFF/CRS tooling (e.g., `rasterio`) when you need georeferenced outputs or to align embeddings with other GIS layers.
- Dequantize on demand to float32 using:

$$
\texttt{emb} = \texttt{emb\_q.astype(float32)} \times \texttt{scales}
$$


## Download and load an example tile

Below we download one embedding tile and its corresponding scales from S3, then dequantize it to float32.


In [ ]:
def load_npy_from_s3(bucket: str, key: str) -> np.ndarray:
    obj = s3.get_object(Bucket=bucket, Key=key)
    return np.load(BytesIO(obj["Body"].read()))

# Example tile (center coordinates). Update lon/lat if this tile is not present in your bucket.
lon, lat = 0.15, 52.05
base = f"grid_{lon:.2f}_{lat:.2f}"

tile_key = f"{year_prefix}{base}.npy"
scales_key = f"{year_prefix}{base}_scales.npy"

embedding_q = load_npy_from_s3(bucket, tile_key)
scales = load_npy_from_s3(bucket, scales_key)

embedding = embedding_q.astype(np.float32) * scales

print("tile_key:", tile_key)
print("scales_key:", scales_key)
print("embedding_q:", embedding_q.shape, embedding_q.dtype)
print("scales:", scales.shape, scales.dtype)
print("embedding:", embedding.shape, embedding.dtype)


In [ ]:
def tile_bounds_from_center(lon_center: float, lat_center: float, tile_size_deg: float = 0.1):
    half = tile_size_deg / 2.0
    return (lon_center - half, lat_center - half, lon_center + half, lat_center + half)

print("Approx tile bounds (min_lon, min_lat, max_lon, max_lat):", tile_bounds_from_center(lon, lat))


## Visualizations

Because each tile is a `(height, width, 128)` array, we can visualize individual channels or summary statistics (e.g., vector norms) as 2D images.


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(embedding[:, :, 0], cmap="viridis")
plt.title("Dequantized Embedding — Channel 0")
plt.axis("off")
plt.colorbar()
plt.show()


In [ ]:
norms = np.linalg.norm(embedding, axis=2)

plt.figure(figsize=(8, 6))
plt.imshow(norms, cmap="magma")
plt.title("Embedding Vector Norms (per pixel)")
plt.axis("off")
plt.colorbar()
plt.show()


In [ ]:
vals = norms.ravel()

plt.figure(figsize=(10, 5))
plt.hist(vals, bins=60, color="#3498db", edgecolor="white")
plt.title("Distribution of Embedding Vector Norms")
plt.xlabel("Norm")
plt.ylabel("Pixel count")
plt.tight_layout()
plt.show()

print(f"Mean norm: {vals.mean():.4f}")
print(f"Median norm: {np.median(vals):.4f}")
print(f"Min norm: {vals.min():.4f}")
print(f"Max norm: {vals.max():.4f}")


## Example analysis

*Are some embedding channels consistently “stronger” than others within a tile?*

A lightweight way to answer this is to compute per-channel mean and standard deviation over all pixels in a tile.


In [ ]:
channel_mean = embedding.mean(axis=(0, 1))
channel_std = embedding.std(axis=(0, 1))

plt.figure(figsize=(12, 4))
plt.plot(channel_mean, label="mean")
plt.plot(channel_std, label="std")
plt.title("Per-channel statistics within one tile")
plt.xlabel("Channel index")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

# Print the top-10 channels by mean magnitude
idx = np.argsort(-np.abs(channel_mean))[:10]
print("Top-10 channels by |mean|:")
for i in idx:
    print(f"channel {int(i):3d}  mean={channel_mean[i]: .4f}  std={channel_std[i]: .4f}")


## Open questions and recommendations

*How transferable are GeoTessera embeddings across regions, years, and downstream tasks (e.g., land cover, change detection, flood mapping)?*

**Recommendations**
- Start with a small region and a single year; confirm you can load and dequantize tiles reliably.
- Build simple baselines first (e.g., linear models on pooled embedding features) before moving to heavier models.
- If you need geospatial alignment with labels (vector/raster), use the landmask GeoTIFF tiles to obtain CRS/transform and ensure consistent reprojection/resampling.
- Consider creating reproducible benchmarks across multiple regions/years to quantify generalization.


## Further resources

- GeoTessera project and Python package documentation: `https://geotessera.readthedocs.io/`
- Tessera foundation model: `https://github.com/ucam-eo/tessera`

## Citation

If you use Tessera / GeoTessera embeddings in academic work, please cite the Tessera paper (see the GeoTessera repository README for the BibTeX entry).